# Adaptive Market-State Learning for Agricultural Commodity ETFs

This notebook implements an initial research framework for agricultural commodity-linked ETFs.

The goal is to integrate **trend**, **momentum**, and **mean-reversion / dislocation** analysis in one adaptive framework using dimensionality reduction and latent-regime learning.

The framework uses:

1. Daily OHLCV agri ETF data, with synthetic fallback.
2. Trend, momentum, mean-reversion, volatility, volume, and cross-ETF context features.
3. PCA for interpretable market-state projection.
4. K-means clustering for latent regime discovery.
5. A walk-forward adaptive policy that chooses `FOLLOW_TREND`, `FOLLOW_MOMENTUM`, `MEAN_REVERT`, or `ABSTAIN`.
6. Baselines, cost sensitivity, and Monte Carlo block-bootstrap robustness.
7. Optional VAE extension.

Research question:

> Can latent market-state representations help distinguish trend-following, momentum, mean-reverting, and uncertain regimes in agricultural commodity ETF markets?

## 0. Leakage discipline

This notebook uses features observed at date `t` and assumes execution at the **next open**. Forward returns are measured from `open[t+1]` to `close[t+horizon]`.

This avoids using the same close both as an input and as a tradable entry price.

For publication-quality work:

- Fit scalers, PCA, clustering, and models only inside the training fold.
- Do not use future OHLCV values in features.
- Treat ETF roll effects, liquidity, tracking difference, and transaction costs as limitations.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 1. Configuration

Default mode is synthetic so the notebook runs immediately. Change `DATA_MODE` to `"yfinance"` or `"csv"` for real data.

CSV format expected: `date, symbol, open, high, low, close, volume`.

In [ ]:
@dataclass
class ResearchConfig:
    DATA_MODE: str = "synthetic"  # synthetic, yfinance, csv
    LOCAL_CSV_PATH: str = "agri_etf_ohlcv.csv"
    START_DATE: str = "2016-01-01"
    END_DATE: str | None = None

    TRADE_TICKERS: tuple[str, ...] = ("DBA", "WEAT", "CORN", "SOYB", "CANE")
    CONTEXT_TICKERS: tuple[str, ...] = ("DBC", "UUP", "USO")

    HORIZON_DAYS: int = 5
    EVENT_ONLY: bool = True
    DISLOCATION_Z_THRESHOLD: float = 1.5
    MOMENTUM_Z_THRESHOLD: float = 1.0
    RANGE_SHOCK_THRESHOLD: float = 1.5

    TRAIN_DAYS: int = 252
    TEST_DAYS: int = 126
    STEP_DAYS: int = 126
    N_PCA_COMPONENTS: int = 3
    N_CLUSTERS: int = 4
    MIN_EDGE_BPS: float = 2.0
    COST_BPS: float = 5.0
    RUN_VAE: bool = False
    RUN_COST_SENSITIVITY: bool = False
    SYNTHETIC_PERIODS: int = 900

cfg = ResearchConfig()
cfg

## 2. Data loading

Initial tradable universe:

- `DBA`: broad agriculture proxy
- `WEAT`: wheat proxy
- `CORN`: corn proxy
- `SOYB`: soybean proxy
- `CANE`: sugar proxy

Context proxies such as broad commodities, US dollar, and energy can be used as market-state inputs.

In [ ]:
def normalize_ohlcv_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rename_map = {}
    for c in df.columns:
        lc = str(c).lower().strip().replace(" ", "_")
        if lc in {"date", "datetime", "timestamp"}:
            rename_map[c] = "date"
        elif lc in {"symbol", "ticker"}:
            rename_map[c] = "symbol"
        elif lc in {"open", "open_price"}:
            rename_map[c] = "open"
        elif lc in {"high", "high_price"}:
            rename_map[c] = "high"
        elif lc in {"low", "low_price"}:
            rename_map[c] = "low"
        elif lc in {"close", "adj_close", "adjusted_close", "close_price"}:
            rename_map[c] = "close"
        elif lc in {"volume", "vol"}:
            rename_map[c] = "volume"
    df = df.rename(columns=rename_map)
    required = ["date", "symbol", "open", "high", "low", "close", "volume"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after normalization: {missing}")
    df = df[required].copy()
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    df["symbol"] = df["symbol"].astype(str).str.upper()
    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["date", "symbol", "open", "high", "low", "close"])
    return df.sort_values(["symbol", "date"]).reset_index(drop=True)


def load_from_yfinance(tickers: list[str], start: str, end: str | None = None) -> pd.DataFrame:
    try:
        import yfinance as yf
    except ImportError as exc:
        raise ImportError("Install yfinance or switch DATA_MODE to synthetic/csv.") from exc
    raw = yf.download(tickers, start=start, end=end, auto_adjust=False, group_by="ticker", progress=False)
    rows = []
    if isinstance(raw.columns, pd.MultiIndex):
        for ticker in tickers:
            if ticker not in raw.columns.get_level_values(0):
                continue
            tmp = raw[ticker].copy()
            tmp = tmp.rename(columns={"Open":"open", "High":"high", "Low":"low", "Close":"close", "Volume":"volume"})
            tmp["date"] = tmp.index
            tmp["symbol"] = ticker
            rows.append(tmp[["date", "symbol", "open", "high", "low", "close", "volume"]])
    else:
        tmp = raw.rename(columns={"Open":"open", "High":"high", "Low":"low", "Close":"close", "Volume":"volume"})
        tmp["date"] = tmp.index
        tmp["symbol"] = tickers[0]
        rows.append(tmp[["date", "symbol", "open", "high", "low", "close", "volume"]])
    if not rows:
        raise ValueError("No data returned. Check tickers/date range/internet.")
    return normalize_ohlcv_columns(pd.concat(rows, ignore_index=True))


def make_synthetic_agri_etf_data(tickers: list[str], start: str = "2016-01-01", periods: int = 1800, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start=start, periods=periods)
    n = len(dates)
    regimes = rng.choice([0, 1, 2], size=n, p=[0.55, 0.25, 0.20])
    agri = rng.normal(0, 0.006, n)
    grain = rng.normal(0, 0.005, n)
    soft = rng.normal(0, 0.006, n)
    dollar = rng.normal(0, 0.0035, n)
    energy = rng.normal(0, 0.006, n)
    for i in range(1, n):
        if regimes[i] == 1:
            agri[i] += 0.25 * agri[i-1]
            grain[i] += 0.20 * grain[i-1]
        elif regimes[i] == 2:
            agri[i] -= 0.30 * agri[i-1]
            grain[i] -= 0.25 * grain[i-1]

    loadings = {
        "DBA":  (0.75, 0.25, 0.25, -0.20, 0.10, 0.006),
        "WEAT": (0.55, 0.75, 0.05, -0.15, 0.05, 0.010),
        "CORN": (0.60, 0.70, 0.05, -0.15, 0.10, 0.009),
        "SOYB": (0.60, 0.65, 0.05, -0.12, 0.12, 0.009),
        "CANE": (0.45, 0.05, 0.80, -0.10, 0.15, 0.011),
        "DBC":  (0.35, 0.20, 0.15, -0.25, 0.55, 0.007),
        "UUP":  (-0.15, -0.05, -0.05, 1.00, -0.05, 0.003),
        "USO":  (0.15, 0.05, 0.10, -0.20, 1.00, 0.014),
    }
    rows = []
    for ticker in tickers:
        l = loadings.get(ticker, (0.5, 0.3, 0.2, -0.1, 0.1, 0.008))
        eps = rng.normal(0, l[5], n)
        ret = l[0]*agri + l[1]*grain + l[2]*soft + l[3]*dollar + l[4]*energy + eps
        shock_idx = rng.choice(np.arange(30, n-30), size=max(5, n//90), replace=False)
        ret[shock_idx] += rng.normal(0, 0.035, len(shock_idx))
        close = 25 * np.exp(np.cumsum(ret))
        open_ = close / (1 + ret) * (1 + rng.normal(0, 0.002, n))
        intraday_range = np.abs(rng.normal(0.012, 0.006, n)) + np.abs(ret) * 0.6
        high = np.maximum(open_, close) * (1 + intraday_range/2)
        low = np.minimum(open_, close) * (1 - intraday_range/2)
        volume = rng.lognormal(mean=12.0, sigma=0.55, size=n).astype(int)
        rows.append(pd.DataFrame({"date": dates, "symbol": ticker, "open": open_, "high": high, "low": low, "close": close, "volume": volume}))
    return normalize_ohlcv_columns(pd.concat(rows, ignore_index=True))


def load_data(cfg: ResearchConfig) -> pd.DataFrame:
    tickers = sorted(set(cfg.TRADE_TICKERS + cfg.CONTEXT_TICKERS))
    if cfg.DATA_MODE == "synthetic":
        return make_synthetic_agri_etf_data(tickers, cfg.START_DATE, periods=cfg.SYNTHETIC_PERIODS, seed=RANDOM_STATE)
    if cfg.DATA_MODE == "yfinance":
        return load_from_yfinance(tickers, cfg.START_DATE, cfg.END_DATE)
    if cfg.DATA_MODE == "csv":
        return normalize_ohlcv_columns(pd.read_csv(cfg.LOCAL_CSV_PATH))
    raise ValueError(f"Unknown DATA_MODE: {cfg.DATA_MODE}")

raw_df = load_data(cfg)
print(raw_df.shape)
raw_df.head()

## 3. Feature engineering

Feature groups:

- trend features,
- momentum features,
- dislocation/mean-reversion features,
- risk and volatility features,
- cross-ETF context features.

In [ ]:
def rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window, min_periods=window).mean()
    avg_loss = loss.rolling(window, min_periods=window).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def add_symbol_features(df: pd.DataFrame, horizons=(1, 3, 5, 10)) -> pd.DataFrame:
    parts = []
    for sym, x in df.sort_values(["symbol", "date"]).groupby("symbol", sort=False):
        x = x.copy().sort_values("date")
        close, open_, high, low, vol = x["close"], x["open"], x["high"], x["low"], x["volume"]
        prev_close = close.shift(1)
        x["daily_return"] = close.pct_change()
        x["log_return"] = np.log(close / prev_close)
        next_open = open_.shift(-1)
        for h in horizons:
            x[f"fwd_return_{h}d"] = close.shift(-h) / next_open - 1

        x["sma_10"] = close.rolling(10, min_periods=10).mean()
        x["sma_20"] = close.rolling(20, min_periods=20).mean()
        x["sma_60"] = close.rolling(60, min_periods=60).mean()
        x["trend_score"] = x["sma_20"] / x["sma_60"] - 1
        x["trend_slope_20"] = x["sma_20"].pct_change(5)
        x["price_vs_sma60"] = close / x["sma_60"] - 1
        x["ret_20"] = close / close.shift(20) - 1
        x["ret_60"] = close / close.shift(60) - 1

        x["ret_5"] = close / close.shift(5) - 1
        x["ret_10"] = close / close.shift(10) - 1
        vol_20 = x["daily_return"].rolling(20, min_periods=20).std()
        x["momentum_score"] = x["ret_5"] / vol_20.replace(0, np.nan)
        x["momentum_20_score"] = x["ret_20"] / (x["daily_return"].rolling(60, min_periods=40).std().replace(0, np.nan) * np.sqrt(20))

        mean20 = close.rolling(20, min_periods=20).mean()
        std20 = close.rolling(20, min_periods=20).std()
        mean60 = close.rolling(60, min_periods=40).mean()
        std60 = close.rolling(60, min_periods=40).std()
        x["dislocation_zscore_20"] = (close - mean20) / std20.replace(0, np.nan)
        x["dislocation_zscore_60"] = (close - mean60) / std60.replace(0, np.nan)
        x["rsi_14"] = rsi(close, 14)
        x["rsi_centered"] = (x["rsi_14"] - 50) / 50

        tr = pd.concat([high-low, (high-prev_close).abs(), (low-prev_close).abs()], axis=1).max(axis=1)
        x["atr_14_pct"] = tr.rolling(14, min_periods=14).mean() / close
        x["range_pct"] = (high - low) / open_.replace(0, np.nan)
        x["range_shock_20"] = x["range_pct"] / x["range_pct"].shift(1).rolling(20, min_periods=15).median()
        hl_log_sq = np.log(high / low).pow(2)
        x["parkinson_vol_20"] = np.sqrt(hl_log_sq.rolling(20, min_periods=20).mean() / (4*np.log(2)))
        x["parkinson_vol_60"] = np.sqrt(hl_log_sq.rolling(60, min_periods=40).mean() / (4*np.log(2)))
        x["volume_zscore_20"] = (np.log1p(vol) - np.log1p(vol).rolling(20, min_periods=20).mean()) / np.log1p(vol).rolling(20, min_periods=20).std()
        rolling_high_60 = close.rolling(60, min_periods=40).max()
        rolling_low_60 = close.rolling(60, min_periods=40).min()
        x["drawdown_60"] = close / rolling_high_60 - 1
        x["distance_from_60d_low"] = close / rolling_low_60 - 1

        x["trend_dir"] = np.sign(x["trend_score"])
        x["momentum_dir"] = np.sign(x["momentum_score"])
        x["mean_reversion_dir"] = -np.sign(x["dislocation_zscore_20"])
        parts.append(x)
    return pd.concat(parts, ignore_index=True)


def add_cross_etf_context(df: pd.DataFrame, trade_tickers: tuple[str, ...], context_tickers: tuple[str, ...]) -> pd.DataFrame:
    ret_wide = df.pivot(index="date", columns="symbol", values="daily_return").sort_index()
    trade_cols = [c for c in trade_tickers if c in ret_wide.columns]
    ctx_cols = [c for c in context_tickers if c in ret_wide.columns]
    context = pd.DataFrame(index=ret_wide.index)
    context["agri_basket_return"] = ret_wide[trade_cols].mean(axis=1)
    context["agri_cross_dispersion"] = ret_wide[trade_cols].std(axis=1)
    context["agri_positive_breadth"] = (ret_wide[trade_cols] > 0).mean(axis=1)
    context["agri_abs_move_median"] = ret_wide[trade_cols].abs().median(axis=1)
    for c in ctx_cols:
        context[f"ctx_{c}_return"] = ret_wide[c]
        context[f"ctx_{c}_return_5d"] = ret_wide[c].rolling(5, min_periods=5).sum()
        context[f"ctx_{c}_vol_20"] = ret_wide[c].rolling(20, min_periods=20).std()
    return df.merge(context.reset_index(), on="date", how="left")

feature_df = add_symbol_features(raw_df, horizons=(1, 3, 5, 10))
feature_df = add_cross_etf_context(feature_df, cfg.TRADE_TICKERS, cfg.CONTEXT_TICKERS)
model_base = feature_df[feature_df["symbol"].isin(cfg.TRADE_TICKERS)].copy()
print(model_base.shape)
model_base.head()

## 4. Event definition and candidate action returns

Candidate actions:

- `FOLLOW_TREND`: follow the slow trend signal.
- `FOLLOW_MOMENTUM`: follow the short-term momentum signal.
- `MEAN_REVERT`: fade the price dislocation.
- `ABSTAIN`: take zero exposure.

In [ ]:
ACTION_COLS = ["FOLLOW_TREND_return", "FOLLOW_MOMENTUM_return", "MEAN_REVERT_return", "ABSTAIN_return"]


def add_event_and_action_returns(df: pd.DataFrame, cfg: ResearchConfig) -> pd.DataFrame:
    df = df.copy()
    fwd_col = f"fwd_return_{cfg.HORIZON_DAYS}d"
    cost = cfg.COST_BPS / 10_000.0
    df["event_flag"] = (
        (df["dislocation_zscore_20"].abs() >= cfg.DISLOCATION_Z_THRESHOLD)
        | (df["momentum_score"].abs() >= cfg.MOMENTUM_Z_THRESHOLD)
        | (df["range_shock_20"] >= cfg.RANGE_SHOCK_THRESHOLD)
    )
    def action_return(direction_col):
        direction = df[direction_col].replace(0, np.nan)
        traded = direction.notna().astype(float)
        return (direction * df[fwd_col]).fillna(0.0) - traded * cost
    df["FOLLOW_TREND_return"] = action_return("trend_dir")
    df["FOLLOW_MOMENTUM_return"] = action_return("momentum_dir")
    df["MEAN_REVERT_return"] = action_return("mean_reversion_dir")
    df["ABSTAIN_return"] = 0.0
    df["oracle_best_action"] = df[ACTION_COLS].idxmax(axis=1).str.replace("_return", "", regex=False)
    df["oracle_best_return"] = df[ACTION_COLS].max(axis=1)
    if cfg.EVENT_ONLY:
        df = df[df["event_flag"]].copy()
    return df.dropna(subset=[fwd_col]).copy()

model_df = add_event_and_action_returns(model_base, cfg)
print("Rows:", len(model_df))
print("Symbols:", model_df["symbol"].nunique())
print("Dates:", model_df["date"].nunique())
model_df[["date", "symbol", "event_flag", "trend_score", "momentum_score", "dislocation_zscore_20", "range_shock_20", "oracle_best_action"]].head()

## 5. Feature columns

These columns are used by PCA, clustering, and the adaptive policy. They exclude all future/action return columns.

In [ ]:
BASE_FEATURE_COLS = [
    "trend_score", "trend_slope_20", "price_vs_sma60", "ret_20", "ret_60",
    "ret_5", "ret_10", "momentum_score", "momentum_20_score",
    "dislocation_zscore_20", "dislocation_zscore_60", "rsi_centered",
    "atr_14_pct", "range_pct", "range_shock_20", "parkinson_vol_20", "parkinson_vol_60",
    "volume_zscore_20", "drawdown_60", "distance_from_60d_low",
    "agri_basket_return", "agri_cross_dispersion", "agri_positive_breadth", "agri_abs_move_median",
]
context_feature_cols = [c for c in model_df.columns if c.startswith("ctx_")]
FEATURE_COLS = [c for c in BASE_FEATURE_COLS + context_feature_cols if c in model_df.columns]

model_df = model_df.dropna(subset=FEATURE_COLS + ACTION_COLS).sort_values(["date", "symbol"]).reset_index(drop=True)
print("Feature count:", len(FEATURE_COLS))
print("Model shape:", model_df.shape)
FEATURE_COLS

## 6. Exploratory PCA and regimes

This global PCA is exploratory only. The walk-forward policy below refits PCA inside each fold.

In [ ]:
def fit_exploratory_pca(df, feature_cols, n_components=3):
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components, random_state=RANDOM_STATE)),
    ])
    z = pipe.fit_transform(df[feature_cols])
    pcs = pd.DataFrame(z, columns=[f"PC{i+1}" for i in range(n_components)], index=df.index)
    return pipe, pcs

pca_pipe, pcs = fit_exploratory_pca(model_df, FEATURE_COLS, cfg.N_PCA_COMPONENTS)
print("Explained variance:", np.round(pca_pipe.named_steps["pca"].explained_variance_ratio_, 4))

kmeans = KMeans(n_clusters=cfg.N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
model_df = pd.concat([model_df, pcs], axis=1)
model_df["pca_regime"] = kmeans.fit_predict(model_df[["PC1", "PC2", "PC3"]])
model_df.groupby("pca_regime")[ACTION_COLS + ["trend_score", "momentum_score", "dislocation_zscore_20", "range_shock_20"]].mean().round(4)

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(model_df["PC1"], model_df["PC2"], c=model_df["pca_regime"], s=12, alpha=0.7)
plt.title("PCA projection of agri ETF market states")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(scatter, label="Regime")
plt.grid(True, alpha=0.3)
plt.show()

## 7. Baselines

The adaptive model must be compared to simple always-on rules.

In [ ]:
def daily_portfolio_returns(df, return_col):
    return df.groupby("date")[return_col].mean().sort_index()


def summarize_daily_returns(daily, label):
    daily = daily.dropna().sort_index()
    if len(daily) == 0:
        return {"strategy": label, "n_days": 0}
    equity = (1 + daily).cumprod()
    dd = equity / equity.cummax() - 1
    ann_ret = daily.mean() * 252
    ann_vol = daily.std(ddof=1) * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol and not np.isnan(ann_vol) else np.nan
    return {
        "strategy": label,
        "n_days": len(daily),
        "mean_daily_return": daily.mean(),
        "ann_return_approx": ann_ret,
        "ann_vol_approx": ann_vol,
        "sharpe_approx": sharpe,
        "cumulative_return": equity.iloc[-1] - 1,
        "max_drawdown": dd.min(),
        "positive_day_rate": (daily > 0).mean(),
    }

baseline_summary_df = pd.DataFrame([
    summarize_daily_returns(daily_portfolio_returns(model_df, col), col.replace("_return", ""))
    for col in ACTION_COLS
])
baseline_summary_df

## 8. Walk-forward adaptive policy

Inside each fold, we fit imputation, scaling, PCA, clustering, and action-return models using only the training window.

In [ ]:
def make_augmented_features(X_train, X_test, n_components, n_clusters, random_state=42):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    pca = PCA(n_components=n_components, random_state=random_state)
    Xtr = scaler.fit_transform(imputer.fit_transform(X_train))
    Xte = scaler.transform(imputer.transform(X_test))
    Ztr = pca.fit_transform(Xtr)
    Zte = pca.transform(Xte)

    # Fast fold-safe regime assignment: fit PC1 quantile thresholds on train, then apply to test.
    # This keeps the walk-forward policy lightweight while still using latent-state bins.
    q = np.linspace(0, 1, n_clusters + 1)[1:-1]
    thresholds = np.quantile(Ztr[:, 0], q)
    tr_regime = np.digitize(Ztr[:, 0], thresholds)
    te_regime = np.digitize(Zte[:, 0], thresholds)

    tr_oh = np.zeros((len(tr_regime), n_clusters))
    te_oh = np.zeros((len(te_regime), n_clusters))
    tr_oh[np.arange(len(tr_regime)), tr_regime] = 1
    te_oh[np.arange(len(te_regime)), te_regime] = 1
    return np.hstack([Xtr, Ztr, tr_oh]), np.hstack([Xte, Zte, te_oh]), te_regime, Zte


def walk_forward_adaptive_policy(df, feature_cols, action_cols, cfg):
    df = df.sort_values(["date", "symbol"]).copy()
    dates = np.array(sorted(df["date"].drop_duplicates()))
    outputs = []
    min_edge = cfg.MIN_EDGE_BPS / 10_000.0
    start = cfg.TRAIN_DAYS
    while start < len(dates):
        train_dates = dates[max(0, start-cfg.TRAIN_DAYS):start]
        test_dates = dates[start:min(start+cfg.TEST_DAYS, len(dates))]
        train = df[df["date"].isin(train_dates)].copy()
        test = df[df["date"].isin(test_dates)].copy()
        if len(train) < 200 or len(test) == 0:
            start += cfg.STEP_DAYS
            continue
        Xtr, Xte, regimes, Zte = make_augmented_features(train[feature_cols], test[feature_cols], cfg.N_PCA_COMPONENTS, cfg.N_CLUSTERS, RANDOM_STATE)
        preds = {}
        for action_col in action_cols:
            m = Ridge(alpha=1.0, random_state=RANDOM_STATE)
            m.fit(Xtr, train[action_col].values)
            preds[action_col.replace("_return", "")] = m.predict(Xte)
        pred_df = pd.DataFrame(preds, index=test.index)
        best_action = pred_df.idxmax(axis=1)
        best_pred = pred_df.max(axis=1)
        chosen_action = best_action.where(best_pred >= min_edge, "ABSTAIN")
        out = test[["date", "symbol"] + action_cols].copy()
        for col in pred_df.columns:
            out[f"pred_{col}"] = pred_df[col]
        out["selected_action"] = chosen_action.values
        out["predicted_edge"] = best_pred.values
        out["walkforward_regime"] = regimes
        for i in range(Zte.shape[1]):
            out[f"fold_PC{i+1}"] = Zte[:, i]
        out["adaptive_return"] = [row.get(f"{row['selected_action']}_return", 0.0) for _, row in out.iterrows()]
        outputs.append(out)
        start += cfg.STEP_DAYS
    if not outputs:
        raise ValueError("No walk-forward output. Reduce TRAIN_DAYS or check sample size.")
    return pd.concat(outputs, ignore_index=True)

adaptive_output = walk_forward_adaptive_policy(model_df, FEATURE_COLS, ACTION_COLS, cfg)
print(adaptive_output.shape)
adaptive_output.head()

In [ ]:
adaptive_daily = daily_portfolio_returns(adaptive_output, "adaptive_return")
adaptive_summary = summarize_daily_returns(adaptive_daily, "ADAPTIVE_POLICY")
comparison = pd.concat([baseline_summary_df, pd.DataFrame([adaptive_summary])], ignore_index=True)
comparison

In [ ]:
adaptive_output["selected_action"].value_counts(normalize=True).rename("share").to_frame()

In [ ]:
plt.figure(figsize=(9, 5))
for action_col in ["FOLLOW_TREND_return", "FOLLOW_MOMENTUM_return", "MEAN_REVERT_return"]:
    (1 + daily_portfolio_returns(model_df, action_col)).cumprod().plot(label=action_col.replace("_return", ""))
(1 + adaptive_daily).cumprod().plot(label="ADAPTIVE_POLICY", linewidth=2)
plt.title("Cumulative return comparison")
plt.ylabel("Growth of $1")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 9. Regime interpretation

A strong research contribution is not just performance, but whether latent regimes correspond to interpretable states.

In [ ]:
regime_interpretation = adaptive_output.groupby("walkforward_regime").agg(
    n=("adaptive_return", "size"),
    mean_adaptive_return=("adaptive_return", "mean"),
    mean_predicted_edge=("predicted_edge", "mean"),
    trend_share=("selected_action", lambda s: (s == "FOLLOW_TREND").mean()),
    momentum_share=("selected_action", lambda s: (s == "FOLLOW_MOMENTUM").mean()),
    mean_revert_share=("selected_action", lambda s: (s == "MEAN_REVERT").mean()),
    abstain_share=("selected_action", lambda s: (s == "ABSTAIN").mean()),
).sort_values("mean_adaptive_return", ascending=False)
regime_interpretation

## 10. Cost sensitivity

A robust result should not depend on one assumed transaction cost.

In [ ]:
def run_cost_sensitivity(cfg, model_base, feature_cols, costs=(0, 2, 5, 10, 20)):
    rows = []
    for cost in costs:
        cfg2 = ResearchConfig(**{**cfg.__dict__, "COST_BPS": float(cost)})
        df2 = add_event_and_action_returns(model_base, cfg2)
        df2 = df2.dropna(subset=feature_cols + ACTION_COLS).sort_values(["date", "symbol"]).reset_index(drop=True)
        try:
            out = walk_forward_adaptive_policy(df2, feature_cols, ACTION_COLS, cfg2)
            summary = summarize_daily_returns(daily_portfolio_returns(out, "adaptive_return"), f"cost_{cost}_bps")
            summary["cost_bps"] = cost
            rows.append(summary)
        except Exception as exc:
            rows.append({"strategy": f"cost_{cost}_bps", "cost_bps": cost, "error": str(exc)})
    return pd.DataFrame(rows)

if cfg.RUN_COST_SENSITIVITY:
    cost_sensitivity = run_cost_sensitivity(cfg, model_base, FEATURE_COLS, costs=(0, 2, 5, 10, 20))
else:
    cost_sensitivity = pd.DataFrame({"note": ["Skipped by default. Set cfg.RUN_COST_SENSITIVITY=True to run."]})
cost_sensitivity

## 11. Monte Carlo block-bootstrap robustness

This resamples blocks of daily portfolio returns to estimate outcome uncertainty.

In [ ]:
def block_bootstrap_paths(daily_returns, n_paths=1000, block_size=10, horizon_days=None, seed=42):
    rng = np.random.default_rng(seed)
    r = daily_returns.dropna().values
    if horizon_days is None:
        horizon_days = len(r)
    if len(r) < block_size + 1:
        raise ValueError("Not enough daily returns for block bootstrap")
    starts = np.arange(0, len(r) - block_size + 1)
    paths = np.zeros((horizon_days, n_paths))
    for p in range(n_paths):
        sampled = []
        while len(sampled) < horizon_days:
            s = rng.choice(starts)
            sampled.extend(r[s:s+block_size])
        sampled = np.array(sampled[:horizon_days])
        paths[:, p] = np.cumprod(1 + sampled) - 1
    return pd.DataFrame(paths)

mc_paths = block_bootstrap_paths(adaptive_daily, n_paths=250, block_size=10, seed=RANDOM_STATE)
terminal = mc_paths.iloc[-1]
mc_summary = pd.Series({
    "terminal_mean": terminal.mean(),
    "terminal_median": terminal.median(),
    "terminal_p05": terminal.quantile(0.05),
    "terminal_p95": terminal.quantile(0.95),
    "prob_positive": (terminal > 0).mean(),
})
mc_summary

In [ ]:
plt.figure(figsize=(9, 5))
for q, label in [(0.05, "5th pct"), (0.50, "median"), (0.95, "95th pct")]:
    mc_paths.quantile(q, axis=1).plot(label=label)
plt.title("Monte Carlo block-bootstrap outcome bands")
plt.xlabel("Simulated day")
plt.ylabel("Cumulative return")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 12. Optional VAE extension

The VAE is optional. Start with PCA + clustering for interpretability. Use VAE later as a nonlinear latent-state extension.

In [ ]:
if cfg.RUN_VAE:
    try:
        import torch
        import torch.nn as nn
        from torch.utils.data import DataLoader, TensorDataset

        class VAE(nn.Module):
            def __init__(self, input_dim, latent_dim=2, hidden_dim=32):
                super().__init__()
                self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
                self.mu = nn.Linear(hidden_dim, latent_dim)
                self.logvar = nn.Linear(hidden_dim, latent_dim)
                self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, input_dim))
            def encode(self, x):
                h = self.encoder(x)
                return self.mu(h), self.logvar(h)
            def reparameterize(self, mu, logvar):
                return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
            def forward(self, x):
                mu, logvar = self.encode(x)
                z = self.reparameterize(mu, logvar)
                return self.decoder(z), mu, logvar

        X = SimpleImputer(strategy="median").fit_transform(model_df[FEATURE_COLS])
        X = StandardScaler().fit_transform(X)
        X_tensor = torch.tensor(X, dtype=torch.float32)
        loader = DataLoader(TensorDataset(X_tensor), batch_size=128, shuffle=True)
        vae = VAE(X.shape[1], latent_dim=2)
        opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
        for epoch in range(30):
            losses = []
            for (batch,) in loader:
                recon, mu, logvar = vae(batch)
                recon_loss = nn.functional.mse_loss(recon, batch)
                kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                loss = recon_loss + 0.01 * kl
                opt.zero_grad(); loss.backward(); opt.step()
                losses.append(loss.item())
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}: loss={np.mean(losses):.6f}")
        with torch.no_grad():
            z, _ = vae.encode(X_tensor)
        model_df["VAE1"] = z[:, 0].numpy()
        model_df["VAE2"] = z[:, 1].numpy()
        plt.figure(figsize=(8, 6))
        plt.scatter(model_df["VAE1"], model_df["VAE2"], c=model_df["pca_regime"], s=12, alpha=0.7)
        plt.title("Optional VAE latent projection")
        plt.xlabel("VAE1"); plt.ylabel("VAE2"); plt.grid(True, alpha=0.3)
        plt.show()
    except ImportError:
        print("PyTorch not installed. Set RUN_VAE=False or install torch.")
else:
    print("VAE section skipped. Set cfg.RUN_VAE=True to run it.")

## 13. Extensions

Recommended next steps:

1. Replace synthetic data with yfinance or your own database.
2. Add intraday features such as VWAP dislocation, intraday range, and intraday volume shock.
3. Add USDA/WASDE event-week flags.
4. Add CFTC positioning percentiles.
5. Compare ETF behaviour against continuous futures where available.
6. Prepare a paper around adaptive market-state learning rather than a single trading rule.